# 03. RAG + Prompt + LLM
이 노트북은 `rag_prompt_llm` 모드를 단독으로 실험합니다.

프로젝트 경로를 설정하고 STREAM 공통 모듈을 import합니다.

In [7]:
from pathlib import Path
import sys

WORKDIR = Path.cwd()
REPO_ROOT = WORKDIR.parent if WORKDIR.name == "notebooks" else WORKDIR
if REPO_ROOT.name == "Root_Stream":
    REPO_ROOT = REPO_ROOT.parent
PROJECT_ROOT = REPO_ROOT / "Root_Stream"

for path_item in (REPO_ROOT, PROJECT_ROOT):
    if str(path_item) not in sys.path:
        sys.path.append(str(path_item))

from Root_Stream.utils.config_loader import load_config
from Root_Stream.utils.path_utils import resolve_path
from Root_Stream.prompts.prompt_manager import PromptManager
from Root_Stream.orchestrator.stream_orchestrator import StreamOrchestrator


사용자 질문을 입력합니다.

In [8]:
user_question = "최근 30일 동안 각 설비별 첫 오류 발생 시각과 마지막 오류 발생 시각, 그리고 총 오류 건수를 함께 보여줘."
user_question


'최근 30일 동안 각 설비별 첫 오류 발생 시각과 마지막 오류 발생 시각, 그리고 총 오류 건수를 함께 보여줘.'

임베딩 -> Chroma 검색 -> 프롬프트 결합 -> SQL 생성 흐름을 실행합니다.

In [9]:
from Root_Stream.services.retrieval.embedding_service import SentenceTransformerEmbeddingService
from Root_Stream.services.retrieval.chroma_retriever import ChromaRetriever
from Root_Stream.services.stream.mode_rag_prompt_llm import _build_context_block

config_path = PROJECT_ROOT / "config" / "config.yaml"
config = load_config(config_path)
project_root = resolve_path(config.get("paths", {}).get("project_root", "."), PROJECT_ROOT)
prompt_file = resolve_path(config["paths"]["prompt_file"], project_root)
prompt_manager = PromptManager(prompt_file)

config["mode"] = "rag_prompt_llm"
config.setdefault("retrieval", {})["enabled"] = True

retrieval_config = config.get("retrieval", {})
embedding_model = str(
    retrieval_config.get(
        "embedding_model",
        config.get("embedding", {}).get("model_name", "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"),
    )
)
chroma_path = resolve_path(retrieval_config["chroma_path"], project_root)
collection_name = str(retrieval_config["collection_name"])
top_k = int(retrieval_config.get("top_k", 3))

embedding_service = SentenceTransformerEmbeddingService(model_name=embedding_model)
retriever = ChromaRetriever(persist_directory=chroma_path, collection_name=collection_name, top_k=top_k)
query_embedding = embedding_service.embed_query(user_question)
preview_contexts = retriever.retrieve(query_embedding)
context_block = _build_context_block(preview_contexts)

prompt_key = "rag_query_generation_prompt"
final_prompt = prompt_manager.render_prompt(
    prompt_key,
    {"question": user_question, "retrieved_context": context_block},
)
print("active_prompt:", prompt_key)
print("----- FINAL PROMPT -----")
print(final_prompt)

orchestrator = StreamOrchestrator(config=config, prompt_manager=prompt_manager, project_root=project_root)
result = orchestrator.run(user_question)
result.to_dict()


[2026-04-09 10:17:21] [INFO] [config_loader.py:load_config:38] 설정 로드 시작: c:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\config\config.yaml
[2026-04-09 10:17:21] [INFO] [config_loader.py:load_config:51] 설정 로드 완료
[2026-04-09 10:17:21] [INFO] [prompt_manager.py:_load_prompts:57] 프롬프트 파일 로드 시작: C:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\prompts\prompt_templates.yaml
[2026-04-09 10:17:21] [INFO] [prompt_manager.py:_load_prompts:70] 프롬프트 파일 로드 완료: prompt_count=3
[2026-04-09 10:17:21] [INFO] [embedding_service.py:_get_model:69] 임베딩 모델 로드 시작: model=sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
[2026-04-09 10:17:24] [INFO] [embedding_service.py:_get_model:73] 임베딩 모델 로드 완료: model=sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
[2026-04-09 10:17:24] [INFO] [embedding_service.py:embed_query:97] 질의 임베딩 시작: text_length=62
[2026-04-09 10:17:24] [INFO] [chroma_retriever.py:_get_collection:79] Chroma 컬렉션 연결 시작: path=C:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Ingest\data\chroma, 

active_prompt: rag_query_generation_prompt
----- FINAL PROMPT -----
너는 Microsoft SQL Server(MSSQL) 전용 Text-to-SQL 질의 생성 전문가다.
사용자의 질문과 검색 컨텍스트를 바탕으로 실제 실행 가능한 MSSQL SELECT 쿼리를 생성하라.

[절대 원칙]
1. 반드시 MSSQL 문법만 사용한다.
2. 반드시 SELECT 쿼리만 생성한다.
3. INSERT, UPDATE, DELETE, MERGE, DROP, ALTER, TRUNCATE, CREATE, EXEC 등 변경 구문은 절대 생성하지 않는다.
4. schema_context에 없는 테이블명, 컬럼명은 절대 사용하지 않는다.
5. retrieved_context는 참고용이며, 실제 SQL 구조는 schema_context를 기준으로 작성한다.
6. 과도한 추측을 하지 않는다.
7. 출력은 SQL 본문만 반환한다.
8. 설명, 코드블록, 주석은 절대 포함하지 않는다.

[MSSQL 규칙]
- LIMIT 대신 TOP을 사용한다.
- 날짜 계산은 DATEADD, DATEDIFF를 사용한다.
- 현재 시각 기준은 GETDATE()를 사용한다.
- 날짜 변환은 CAST 또는 CONVERT를 사용한다.
- 집계 쿼리에서는 GROUP BY를 정확히 작성한다.
- TOP 사용 시 필요한 경우 ORDER BY를 함께 사용한다.

[사용자 질문]
최근 30일 동안 각 설비별 첫 오류 발생 시각과 마지막 오류 발생 시각, 그리고 총 오류 건수를 함께 보여줘.

[스키마 정보]


[검색 참고 컨텍스트]
[1] chunk_id=644bd73a26fb279f_chunk_0002, score=1.0046265125274658
Alarm identifier/code
    ALARMTEXT   varchar(100) NULL,                      -- Alarm description text
    FILENAME    varch

[2026-04-09 10:17:28] [INFO] [embedding_service.py:_get_model:73] 임베딩 모델 로드 완료: model=sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
[2026-04-09 10:17:28] [INFO] [embedding_service.py:embed_query:97] 질의 임베딩 시작: text_length=62
[2026-04-09 10:17:28] [INFO] [chroma_retriever.py:_get_collection:79] Chroma 컬렉션 연결 시작: path=C:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Ingest\data\chroma, collection=doc_chunks
[2026-04-09 10:17:28] [INFO] [chroma_retriever.py:retrieve:121] RAG 검색 시작: top_k=3
[2026-04-09 10:17:28] [INFO] [chroma_retriever.py:retrieve:165] RAG 검색 완료: result_count=3, sample_ids=644bd73a26fb279f_chunk_0002, 644bd73a26fb279f_chunk_0006, 644bd73a26fb279f_chunk_0009
[2026-04-09 10:17:28] [INFO] [ollama_client.py:generate:122] Ollama 호출 시작: model=qwen2.5:14b, endpoint=http://192.168.0.74:11434/api/chat
[2026-04-09 10:17:31] [INFO] [ollama_client.py:generate:170] Ollama 호출 완료: output_length=261
[2026-04-09 10:17:31] [INFO] [mode_rag_prompt_llm.py:run_rag_prompt_llm_mode:166] mod

{'mode': 'rag_prompt_llm',
 'question': '최근 30일 동안 각 설비별 첫 오류 발생 시각과 마지막 오류 발생 시각, 그리고 총 오류 건수를 함께 보여줘.',
 'query': "```sql\nSELECT \n    EQPID,\n    MIN(EVENTTIME) AS FIRST_ERROR_TIME,\n    MAX(EVENTTIME) AS LAST_ERROR_TIME,\n    COUNT(*) AS ERROR_COUNT\nFROM \n    EVENT_LOG_DATA\nWHERE \n    ACTION = 'ERROR' AND \n    EVENTTIME >= DATEADD(DAY, -30, GETDATE())\nGROUP BY \n    EQPID;\n```",
 'llm_provider': 'ollama',
 'prompt_key': 'rag_query_generation_prompt',
 'retrieved_contexts': [{'chunk_id': '644bd73a26fb279f_chunk_0002',
   'text': 'Alarm identifier/code\n    ALARMTEXT   varchar(100) NULL,                      -- Alarm description text\n    FILENAME    varchar(64)  NULL,                      -- Source file name or ingest file name\n    CREATETIME  datetime2(7) NULL DEFAULT getdate(),    -- Row creation / insert timestamp\n\n    CONSTRAINT ERROR_LOG_DATA_PK PRIMARY KEY (\n        EQPID,\n        MODULEID,\n        EVENTTIME,\n        UNIT,\n        ACTION,\n        ALARMID\n    

검색된 RAG 컨텍스트를 확인합니다.

In [10]:
[item.to_dict() for item in result.retrieved_contexts]


[{'chunk_id': '644bd73a26fb279f_chunk_0002',
  'text': 'Alarm identifier/code\n    ALARMTEXT   varchar(100) NULL,                      -- Alarm description text\n    FILENAME    varchar(64)  NULL,                      -- Source file name or ingest file name\n    CREATETIME  datetime2(7) NULL DEFAULT getdate(),    -- Row creation / insert timestamp\n\n    CONSTRAINT ERROR_LOG_DATA_PK PRIMARY KEY (\n        EQPID,\n        MODULEID,\n        EVENTTIME,\n        UNIT,\n        ACTION,\n        ALARMID\n    )\n);\n\n-- Recommended semantic notes for retrieval:\n-- 1) EVENTTIME is usually the main time filter column for event-range questions.\n-- 2) CREATETIME is the record creation time, not necessarily the event occurrence time.\n-- 3) ACTION and ALARMID together are key dimensions for grouping or deduplication.\n\n\n-- ===== EVENT_LOG_DATA_cleaned.sql =====\n-- Ingest',
  'score': 1.0046265125274658,
  'metadata': {'file_size': 7043,
   'source_path': 'C:\\Users\\김민한\\Desktop\\개발\\DB_to_

## MSSQL 실행 단계
위에서 생성된 `result.query`를 공통 SQL 실행 서비스로 안전하게 실행합니다.

In [11]:
from IPython.display import display

from Root_Stream.services.sql.sql_executor_service import SqlExecutorService
from Root_Stream.services.sql.sql_execution_integration import (
    build_execution_payload,
    run_generated_sql_with_executor,
)

# MSSQL connection (IP / PORT / DB / ID / PW)
db_host = "192.168.0.11"
db_port = 1433
db_name = "SVM3PRISM"
db_user = "prismadm"
db_password = "prism@2025"

database_config = config.setdefault("database", {})
database_config.update(
    {
        "type": "mssql",
        "host": db_host,
        "port": db_port,
        "database": db_name,
        "username": db_user,
        "password": db_password,
        "driver": database_config.get("driver", "ODBC Driver 17 for SQL Server"),
        "encrypt": bool(database_config.get("encrypt", False)),
        "trust_server_certificate": bool(database_config.get("trust_server_certificate", True)),
        "timeout": int(database_config.get("timeout", 30)),
    }
)

sql_config = config.setdefault("sql", {})
sql_config.setdefault("allow_only_select", True)
sql_config.setdefault("max_rows", 1000)

if "result" not in locals():
    raise ValueError("먼저 SQL 생성 셀을 실행해 result를 만든 뒤 이 셀을 실행하세요.")

executor = SqlExecutorService.from_config(config)
try:
    generated_sql = result.query
    df = run_generated_sql_with_executor(generated_sql=generated_sql, executor=executor)
    execution_payload = build_execution_payload(df)
finally:
    executor.close()

print(f"조회 row_count: {execution_payload['row_count']}")
print("조회 결과 표(최대 20행 미리보기)")
display(df.head(20))
execution_payload


[2026-04-09 10:17:55] [INFO] [sql_executor_service.py:from_config:59] SQL 실행 서비스 생성 완료: allow_only_select=True, max_rows=1000
[2026-04-09 10:17:55] [INFO] [sql_execution_integration.py:run_generated_sql_with_executor:44] 생성 SQL 실행 연결 시작
[2026-04-09 10:17:55] [INFO] [sql_executor_service.py:run_query:81] SQL 실행 요청 수신
[2026-04-09 10:17:55] [INFO] [sql_guard.py:validate_query_sql:60] SQL 검증 시작
[2026-04-09 10:17:55] [INFO] [sql_guard.py:_normalize_sql_text:85] SQL code block wrapper removed before validation.
[2026-04-09 10:17:55] [INFO] [sql_guard.py:validate_query_sql:74] SQL 검증 완료
[2026-04-09 10:17:55] [INFO] [sql_executor_service.py:run_query:83] MSSQL 조회 실행 시작
[2026-04-09 10:17:55] [INFO] [mssql_client.py:execute_query:110] MSSQL 조회 시작: host=192.168.0.11, database=SVM3PRISM
[2026-04-09 10:17:55] [ERROR] [mssql_client.py:execute_query:116] MSSQL 조회 실패
Traceback (most recent call last):
  File "c:\chat\Lib\site-packages\sqlalchemy\engine\base.py", line 1967, in _exec_single_context
    

RuntimeError: MSSQL 조회 실행 중 오류가 발생했습니다.